# Exercise: Column Transformers and Pipelines

## Do You Want to Build a Snowman?

Prepare the same winter data again, this time with a `ColumnTransformer` and pipelines, as in notebook 03.

<img src="assets/olaf.jpeg">

## Learning Objectives

At the end of this exercise, you should be able to:

- Chain two preprocessing steps for one column in a `Pipeline`.
- Combine the transformations of all the columns in one `ColumnTransformer`.
- Transform the test data with the fitted `ColumnTransformer` in a single call.

## Setup

Documentation: [`set_config`](https://scikit-learn.org/stable/modules/generated/sklearn.set_config.html).

In [ ]:
import pandas as pd
from sklearn import set_config
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder

# Transformers return pandas DataFrames instead of NumPy arrays
set_config(transform_output="pandas")

## The Data

Each row is one winter day: the temperature, what you had for lunch and dinner, whether it snowed or rained, and the height of the snowman you built. The target is `height_snowman_cm`.

In [ ]:
data = {
    "temp": [-3, 5, 0, 7, 3, -1, 1, None, -6, 3, 0, -1, None, -2],
    "lunch": [
        "soup",
        "sandwich",
        "soup",
        "burger",
        "sandwich",
        "soup",
        "cereal",
        "salad",
        "sandwich",
        "burger",
        "soup",
        "cereal",
        "burger",
        "soup",
    ],
    "dinner": [
        "pizza",
        "pizza",
        "noodles",
        None,
        "fishsticks",
        "pizza",
        None,
        "fishsticks",
        "noodles",
        "pizza",
        None,
        "pizza",
        "fishsticks",
        "pizza",
    ],
    "precipitation": [
        "yes",
        "no",
        "yes",
        "yes",
        "yes",
        "yes",
        "no",
        "yes",
        "yes",
        "no",
        "yes",
        "yes",
        "yes",
        "no",
    ],
    "height_snowman_cm": [100, 0, 75, 0, 20, 25, 0, 35, 170, 0, 85, 85, 45, 0],
}

df_train = pd.DataFrame(data=data)
df_train

The test set below contains four new days. Keep it aside until you have fitted the transformers on the training data: you will then transform it with those fitted transformers.

In [ ]:
# Four new days, kept aside to check the fitted transformers on unseen data
test_data = {
    "temp": [2, None, -4, 6],
    "lunch": ["soup", "burger", "pasta", "sandwich"],
    "dinner": ["pizza", "noodles", None, "fishsticks"],
    "precipitation": ["yes", "no", "yes", "no"],
    "height_snowman_cm": [40, 0, 110, 0],
}

df_test = pd.DataFrame(data=test_data)
df_test

## 1. Separate the Features and the Target

Split `df_train` into `X_train` and `y_train`, and `df_test` into `X_test` and `y_test`.

In [ ]:
target = "height_snowman_cm"

# TODO: features and target for the training and the test data
X_train, y_train = ...
X_test, y_test = ...

## 2. Build a Pipeline for Each Column That Needs Two Steps

`temp` needs imputing **and** scaling, and `dinner` needs imputing **and** encoding. A [`Pipeline`](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html) runs such steps one after the other. Build one pipeline for each of the two columns, with each step as a `(name, transformer)` tuple. Use the same imputation strategies and encoder settings as in the previous exercise.

Documentation: [`SimpleImputer`](https://scikit-learn.org/stable/modules/generated/sklearn.impute.SimpleImputer.html), [`OneHotEncoder`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html), [`MinMaxScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MinMaxScaler.html).

In [ ]:
# TODO: temp: impute with the median, then scale with MinMaxScaler
temp_pipeline = Pipeline(steps=[("imputer", ...), ("scaler", ...)])

# TODO: dinner: impute with the most frequent value, then one-hot encode
#       (drop="first", handle_unknown="ignore", sparse_output=False)
dinner_pipeline = ...

## 3. Combine Them in a `ColumnTransformer`

A [`ColumnTransformer`](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html) sends each group of columns to its own transformer. The `temp` entry is already given. Add an entry for the `dinner` pipeline, and one for `lunch` and `precipitation`, which only need a `OneHotEncoder` with the same settings as for `dinner`. `remainder="drop"` discards every column that no entry lists.

In [ ]:
# TODO: one entry per column group: (name, transformer, columns)
feature_engineering = ColumnTransformer(
    transformers=[
        ("temp", temp_pipeline, ["temp"]),
        ...,
        ...,
    ],
    remainder="drop",
)

## 4. Fit and Transform the Training Data

Fit the `ColumnTransformer` on `X_train` and transform it in one step with `.fit_transform()`. Every imputer, encoder and scaler inside it learns from the training data only.

In [ ]:
# TODO: fit the ColumnTransformer on X_train and transform X_train in one step
X_train_fe = ...
X_train_fe

## 5. Transform the Test Data

Transform `X_test` with the fitted `ColumnTransformer`, using only `.transform()`. A single call now applies every step, in the right order, with the values learned from the training data. Compare `X_test_fe` with the result of the previous exercise.

In [ ]:
# TODO: transform X_test with the fitted ColumnTransformer (no fitting!)
X_test_fe = ...
X_test_fe

## Bonus: Add the Model

Put the `ColumnTransformer` and a [`LinearRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html) into one pipeline, fit it on the training data and predict the snowman heights for the four test days. With only 14 training days, the predictions will not be accurate. The aim is to see the whole workflow, from raw data to predictions, run in two calls: `.fit()` and `.predict()`.

In [ ]:
# TODO: a Pipeline with the steps ("feature_engineering", ...) and ("model", LinearRegression())
snowman_model = ...

# TODO: fit on the training data, then predict the test days
...

## Summary

In this exercise you:

- Chained imputation with scaling, and imputation with encoding, in a `Pipeline` for each column that needs two steps.
- Combined all the column groups in one `ColumnTransformer`.
- Fitted it on the training data and transformed the test data with a single `.transform()` call.
- Wrapped the preprocessing and a model in one pipeline (bonus).